# Lab Assignment 12: Interactive Visualizations and Dashboards
## DS 6001

### Instructions
Please answer the following questions as completely as possible using text, code, and the results of code as needed. Format your answers in a Jupyter notebook. To receive full credit, make sure you address every part of the problem, and make sure your document is formatted in a clean and professional way.

**Note: Your PDF output will probably cutoff some of the figures you create. That is fine. Under different circumstances we wouldn't ask you to share your HTML-enabled products with PDFs, but we are forced by Gradescope to do that. All we are looking for is the correct code and some evidence that the code ran successfully.**

## Problem 0
Load the Conda environment you built in module 1 as the kernel of this notebook. Then import the following packages:

In [306]:
import numpy as np
import pandas as pd

# Plotly modules/methods/settings
import plotly.graph_objects as go
import plotly.express as px
import plotly.figure_factory as ff
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True) # enables display of plotly figures in HTML/PDF notebooks

# Dash modules/methods/settings
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css'] # Controls default visual appearance of the dashboard

For this lab, we will be working with the 2019 General Social Survey one last time.

In [ ]:
%%capture
gss = pd.read_csv("https://github.com/jkropko/DS-6001/raw/master/localdata/gss2018.csv",
                 encoding='cp1252', na_values=['IAP','IAP,DK,NA,uncodeable', 'NOT SURE',
                                               'DK', 'IAP, DK, NA, uncodeable', '.a', "CAN'T CHOOSE"])
                    
# gss = pd.read_csv("gss2018.csv", encoding='cp1252', na_values=['IAP','IAP,DK,NA,uncodeable', 'NOT SURE',
                                            #    'DK', 'IAP, DK, NA, uncodeable', '.a', "CAN'T CHOOSE"])
                                               

Here is code that cleans the data and gets it ready to be used for data visualizations:

In [308]:
mycols = ['id', 'wtss', 'sex', 'educ', 'region', 'age', 'coninc',
          'prestg10', 'mapres10', 'papres10', 'sei10', 'satjob',
          'fechld', 'fefam', 'fepol', 'fepresch', 'meovrwrk'] 
gss_clean = gss[mycols]
gss_clean = gss_clean.rename({'wtss':'weight', 
                              'educ':'education', 
                              'coninc':'income', 
                              'prestg10':'job_prestige',
                              'mapres10':'mother_job_prestige', 
                              'papres10':'father_job_prestige', 
                              'sei10':'socioeconomic_index', 
                              'fechld':'relationship', 
                              'fefam':'male_breadwinner', 
                              'fehire':'hire_women', 
                              'fejobaff':'preference_hire_women', 
                              'fepol':'men_bettersuited', 
                              'fepresch':'child_suffer',
                              'meovrwrk':'men_overwork'},axis=1)
gss_clean.age = gss_clean.age.replace({'89 or older':'89'})
gss_clean.age = gss_clean.age.astype('float')

In [309]:
gss_clean

,id,weight,sex,education,region,age,income,job_prestige,mother_job_prestige,father_job_prestige,socioeconomic_index,satjob,relationship,male_breadwinner,men_bettersuited,child_suffer,men_overwork
0,1,2.357493,male,14.0,new england,43.0,NaN,47.0,31.0,45.0,65.3,very satisfied,strongly agree,disagree,agree,strongly disagree,agree
1,2,0.942997,female,10.0,new england,74.0,22782.5000,22.0,32.0,39.0,14.8,NaN,NaN,NaN,NaN,NaN,NaN
2,3,0.942997,male,16.0,new england,42.0,112160.0000,61.0,32.0,72.0,83.4,mod. satisfied,strongly agree,disagree,disagree,disagree,disagree
3,4,0.942997,female,16.0,new england,63.0,158201.8412,59.0,NaN,39.0,69.3,very satisfied,agree,disagree,disagree,disagree,neither agree nor disagree
4,5,0.942997,male,18.0,new england,71.0,158201.8412,53.0,35.0,45.0,68.6,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2343,2344,0.471499,female,12.0,new england,37.0,NaN,47.0,31.0,72.0,38.8,mod. satisfied,disagree,strongly disagree,disagree,strongly disagree,disagree
2344,2345,0.942997,female,12.0,new england,75.0,22782.5000,28.0,NaN,27.0,21.6,very satisfied,strongly agree,disagree,disagree,disagree,disagree
2345,2346,0.942997,female,12.0,new england,67.0,70100.0000,40.0,45.0,53.0,41.8,NaN,NaN,NaN,NaN,NaN,NaN
2346,2347,0.942997,male,16.0,new england,72.0,38555.0000,47.0,53.0,50.0,62.7,NaN,disagree,agree,disagree,strongly agree,agree


The `gss_clean` dataframe now contains the following features:

* `id` - a numeric unique ID for each person who responded to the survey
* `weight` - survey sample weights
* `sex` - male or female
* `education` - years of formal education
* `region` - region of the country where the respondent lives
* `age` - age
* `income` - the respondent's personal annual income
* `job_prestige` - the respondent's occupational prestige score, as measured by the GSS using the methodology described above
* `mother_job_prestige` - the respondent's mother's occupational prestige score, as measured by the GSS using the methodology described above
* `father_job_prestige` -the respondent's father's occupational prestige score, as measured by the GSS using the methodology described above
* `socioeconomic_index` - an index measuring the respondent's socioeconomic status
* `satjob` - responses to "On the whole, how satisfied are you with the work you do?"
* `relationship` - agree or disagree with: "A working mother can establish just as warm and secure a relationship with her children as a mother who does not work."
* `male_breadwinner` - agree or disagree with: "It is much better for everyone involved if the man is the achiever outside the home and the woman takes care of the home and family."
* `men_bettersuited` - agree or disagree with: "Most men are better suited emotionally for politics than are most women."
* `child_suffer` - agree or disagree with: "A preschool child is likely to suffer if his or her mother works."
* `men_overwork` - agree or disagree with: "Family life often suffers because men concentrate too much on their work."

## Problem 1
Our goal in this lab is to build a dashboard that presents our findings from the GSS. A dashboard is meant to be shared with an audience, whether that audience consists of managers, clients, potential employers, or the general public. So we need to provide context for our results. One way to provide context is to write text using markdown code.

Find at least two websites that discuss the gender wage gap, and write a short paragraph in markdown code summarizing what these sources tell us. Include hyperlinks to the websites you use. Some potential sources (though you can find your own sources as well) are:

* https://en.wikipedia.org/wiki/Gender_pay_gap

* https://www.pewresearch.org/short-reads/2025/03/04/gender-pay-gap-in-us-has-narrowed-slightly-over-2-decades/

* https://news.darden.virginia.edu/2024/04/04/why-the-gender-pay-gap-persists-in-american-businesses/ 

Then write another short paragraph describing what the GSS is, what the data contain, how it was collected, and/or other information that you think your audience ought to know. A good starting point for information about the GSS is here: https://gss.norc.org/us/en/gss/about-the-gss.html

Then save the text as a Python string so that you can use the markdown code in your dashboard later.

It is easy to copy-and-paste text from a website, but please don't: that communicates a lack of care to your audience. It is also easy to simply prompt an LLM to generate this text, but that is prone to hallucinations. Please summarize the text from websites in your own words, and if you use an LLM (it may be wiser not to), then double check specific facts to make sure they are accurate.

[5 points]

<strong>

Though there have been gains in the past few decades, the pay gap between men and women in the United States still persists. The smallest difference is observed at the lowest wages, where minimum wage requirements normalize how little many workers are paid ([Economic Polity Institute](https://www.epi.org/blog/gender-pay-gap-2024/)). Additionally, the pay gap differs at different ages; according to ([Equal Pay Today](https://www.equalpaytoday.org/gender-pay-gap-statistics/)), for individuals between 25 and 34 years old, women make 95% of what men do. Globally, women make 23% percent less than men do, whereas in the United States the difference is only 15% . Despite these and many more differences, one thing persists accross age, race, location, and occupation: women are, on average paid less. 

Since 1972, the [General Social Survey](https://gss.norc.org/about-the-gss.html) by the National Opinion Research Ceneter at the University of Chicago has served as a premier source of data on the state of American society. By collecting and providing this data to the public, NORC enables public access to a trove of data from which we can put our finger to the pulse of the politics, mental health, opinions, prejudices, and priorities of Americans. In this dashboard, we use that data to demonstrate the state of the gender pay gap in the United States.
</strong>


In [310]:
dashboard_description = '''
Though there have been gains in the past few decades, the pay gap between men and women in the United States still persists. The smallest difference is observed at the lowest wages, where minimum wage requirements normalize how little many workers are paid ([Economic Polity Institute](https://www.epi.org/blog/gender-pay-gap-2024/)). Additionally, the pay gap differs at different ages; according to ([Equal Pay Today](https://www.equalpaytoday.org/gender-pay-gap-statistics/)), for individuals between 25 and 34 years old, women make 95% of what men do. Globally, women make 23% percent less than men do, whereas in the United States the difference is only 15% . Despite these and many more differences, one thing persists accross age, race, location, and occupation: women are, on average paid less. 

Since 1972, the General Social Survey by the National Opinion Research Ceneter at the University of Chicago has served as a premier source of data on the state of American society. By collecting and providing this data to the public, NORC enables public access to a trove of data from which we can put our finger to the pulse of the politics, mental health, opinions, prejudices, and priorities of Americans. In this dashboard, we use that data to demonstrate the state of the gender pay gap in the United States.'''

## Problem 2
Generate a table that shows the mean income, occupational prestige, socioeconomic index, and years of education for men and for women. Use the `ff.create_table()` method to display a web-enabled version of this table. This table is for presentation purposes, so round every column to two decimal places and use more presentable column names. [15 points]

In [312]:
gss_table = gss_clean[['income', 'socioeconomic_index','job_prestige', 'education', 'sex']]
gss_table = gss_table.groupby('sex').agg('mean').round(2).reset_index()
gss_table = gss_table.rename({'income':'Mean Income',
                  'sex':'Sex',
                  'socioeconomic_index':'Mean Socioeconomic Index',
                  'job_prestige':'Mean Occupational Prestige',
                  'education':'Mean Years of Education'}, axis=1)
ff.create_table(gss_table)

## Problem 3
Use plotly express to create the figure for this problem, as well as the figures for problems 4, 5, and 6.

Create an interactive barplot that shows the number of men and women who respond with each level of agreement to `male_breadwinner`. Write presentable labels for the x and y-axes, but don't bother with a title because we will be using a subtitle on the dashboard for this graphic. [15 points]

In [313]:
gss_barplot = gss_clean.groupby('sex')['male_breadwinner'].value_counts().reset_index()
gss_barplot

,sex,male_breadwinner,count
0,female,disagree,377
1,female,strongly disagree,286
2,female,agree,152
3,female,strongly agree,48
4,male,disagree,337
5,male,agree,158
6,male,strongly disagree,147
7,male,strongly agree,40


In [314]:
barplot = px.bar(gss_barplot, x='sex', y='male_breadwinner', color='male_breadwinner', barmode='group', labels={
    'sex': 'Sex',
    'male_breadwinner': 'Count Responses'
}, text='count')

barplot.update(layout=dict(title=dict(x=0.5)))

## Problem 4
Create an interactive scatterplot with `job_prestige` on the x-axis and `income` on the y-axis. Color code the points by `sex` and make sure that the figure includes a legend for these colors. Also include two best-fit lines, one for men and one for women. Finally, include hover data that shows us the values of `education` and `socioeconomic_index` for any point the mouse hovers over. Write presentable labels for the x and y-axes, but don't bother with a title because we will be using a subtitle on the dashboard for this graphic. 

If you see an error that says the package "statsmodels" is not installed, add it to your conda environment via the terminal by activating the environment then typing `conda install statsmodels`. [15 points]

In [315]:
scatter = px.scatter(gss_clean,
                    x='job_prestige',
                    y='income',
                    color='sex',
                    trendline='ols', 
                    hover_data=['education', 'socioeconomic_index'],
                    labels={
                        'income': 'Income',
                        'job_prestige': 'Occupational Prestige',
                        'education': 'Education',
                        'socioeconomic_index': 'Socioeconomic Index'
                    })
scatter.show()

## Problem 5
Create two interactive box plots: one that shows the distribution of `income` for men and for women, and one that shows the distribution of `job_prestige` for men and for women. Write presentable labels for the axis that contains `income` or `job_prestige` and remove the label for `sex`. Also, turn off the legend. Don't bother with titles because we will be using subtitles on the dashboard for these graphics. [15 points]

In [292]:
gss_box_1 = gss_clean[['income', 'sex']].dropna().sort_values(by='sex')
gss_box_1 = gss_box_1.rename({'sex': 'Sex', 'income': 'Income'}, axis=1)
boxplot1 = px.box(gss_box_1, x='Sex', y='Income', color='Sex')
boxplot1.show()

In [293]:
gss_box_2 = gss_clean[['job_prestige', 'sex']].dropna().sort_values(by='sex')
gss_box_2 = gss_box_2.rename({'sex': 'Sex', 'job_prestige': 'Job Prestige'}, axis=1)
boxplot2 = px.box(gss_box_2, x='Sex', y='Job Prestige', color='Sex')
boxplot2.show()

## Problem 6
Create a new dataframe that contains only `income`, `sex`, and `job_prestige`. Then create a new feature in this dataframe that breaks `job_prestige` into six categories with equally sized ranges. Finally, drop all rows with any missing values in this dataframe.

Then create a facet grid with three rows and two columns in which each cell contains an interactive box plot comparing the income distributions of men and women for each of these new categories. 

(If you want men to be represented by blue and women by red, you can include `color_discrete_map = {'male':'blue', 'female':'red'}` in your plotting function. Or use different colors if you want!) [15 points]

In [316]:
data6 = gss_clean[['income', 'sex', 'job_prestige']]
data6['job_prestige_group'] = pd.cut(data6['job_prestige'], bins=6, labels=['Very Low', 'Low', 'Medium Low', 'Medium High', 'High', 'Very High'])
data6 = data6.dropna()
data6 = data6.sort_values(by='job_prestige_group')

facet_box = px.box(data6, 
    facet_col='job_prestige_group',
    x='sex', 
    y='income', 
    facet_col_wrap=2, 
    color='sex', 
    height=900, 
    color_discrete_map={'male': 'green', 'female': 'orange'},
    labels={
        'job_prestige_group': 'Prestige'
    }
)

facet_box.show()


## Problem 7
Create a dashboard that displays the following elements:

* A descriptive title

* The markdown text you wrote in problem 1

* The table you made in problem 2

* The barplot you made in problem 3

* The scatterplot you made in problem 4

* The two boxplots you made in problem 5 side-by-side

* The faceted boxplots you made in problem 6

* Subtitles for all of the above elements

Note: the `dash()` method will display the dashboard directly in your notebook. You do not need to use screenshots to show it, or do anything other than what `dash()` does by default. My textbook uses `JupyterDash`, but this package is now deprecated as `dash()` has now built-in this functionality.

Any working dashboard that displays all of the above elements will receive full credit. [20 points]

In [303]:
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
title = "Gender Pay Gap"

In [ ]:
app = dash.Dash(external_stylesheets=external_stylesheets)

app.layout = html.Div([
    html.H1(title, style=dict(textAlign="center")),
    dcc.Markdown(children = dashboard_description),
    html.Div([
        html.Div([
            html.H2("Count of Temperature Responses by Sex"),
            html.P('"It is much better for everyone involved if the man is the achiever outside the home and the woman takes care of the home and family."'),
            dcc.Graph(id="barplot", figure=barplot)], 
            style=dict(flex=1, padding="0 10px", minWidth=0, textAlign="center")
        ),
        html.Div([
            html.H2("Occupational Prestige vs. Income by Sex", style=dict(textAlign="center")),
            dcc.Graph(id="scatterplot", figure=scatter)], 
            style=dict(flex=1, padding="0 10px", minWidth=0)
        ),
    ], id="graphs-wrapper1", style=dict(display='flex', flex='wrap', alignItems='stretch')),
    html.H2("Distribution of Features by Sex", style=dict(textAlign="center")),
    html.Div([
        html.Div([
            html.H3("Income", style=dict(textAlign="center")),
            dcc.Graph(id="boxplot1", figure=boxplot1)], 
            style=dict(flex=1, padding="0 10px", minWidth=0)
        ),
        html.Div([
            html.H3("Occupational Prestige", style=dict(textAlign="center")),
            dcc.Graph(id="boxplot2", figure=boxplot2)], 
            style=dict(flex=1, padding="0 10px", minWidth=0)
        )
    ], id="graphs-wrapper2", style=dict(display='flex', flex='wrap', alignItems='stretch')),
    html.Div([
        html.Div([
            html.H2("Distributions by Sex and Occupational Prestige", style=dict(textAlign="center")),
            dcc.Graph(id="facet_box", figure=facet_box)], 
            style=dict(flex=1, padding="0 10px", minWidth=0, height="100%")
        )
    ], id="graphs-wrapper3", style=dict(display='flex', flex='wrap', alignItems='stretch', height="10em"))
], style=dict(fontFamily="sans-serif"))

app.run_server(debug=True, port=8051)



## Extra Credit (up to 50 bonus points)
Dashboards are all about good design, functionality, and accessability. For this extra credit problem, create another version of the dashboard you built for problem 7, but take extra steps to improve the appearance of the dashboard, add user-inputs, and host it on the internet with its own URL.

**Challenge 1**: Be creative and use a layout that significantly departs from the one used for the ANES data in the module 12 notebook. A good place to look for inspiration is the [Dash gallery](https://dash-gallery.plotly.host/Portal/). We will award up to 15 bonus points for creativity, novelty, and style.

**Challenge 2**: Alter the barplot from problem 3 to include user inputs. Create two dropdown menus on the dashboard. The first one should allow a user to display bars for the categories of `satjob`, `relationship`, `male_breadwinner`, `men_bettersuited`, `child_suffer`, or `men_overwork`. The second one should allow a user to group the bars by `sex`, `region`, or `education`. After choosing a feature for the bars and one for the grouping, program the barplot to update automatically to display the user-inputted features. Five bonus points will be awarded for a good effort, and 15 bonus points will be awarded for a working user-input barplot in the dashboard.

**Challenge 3**: Follow these steps to host the dashboard publicly on PythonAnywhere: https://docs.google.com/document/d/1lYxsRQ_J0llM5Ztk0CN4c5JQkzlyr8dv6qMjcfLsMh0/edit?usp=sharing 20 bonus points will be awarded for a working PythonAnywhere link.